In [1]:
import tqdm as notebook_tqdm
from datasets import Dataset
import pandas as pd
import math
import torch
from transformers import XLNetTokenizer, XLNetLMHeadModel, Trainer, TrainingArguments, pipeline

/home/shady/Desktop/project/venv/lib64/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-04-29 13:03:22.331173: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-29 13:03:22.339314: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-29 13:03:22.349110: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-29 13:03

# XLNet Fine-tuning with Permutation Language Modeling

XLNet uses permutation language modeling, which is different from traditional autoregressive language models. To properly fine-tune XLNet, we need to correctly handle:

1. **perm_mask**: Controls which tokens a position can attend to, simulating different permutations
2. **target_mapping**: Specifies which positions to predict

In this implementation, we'll set up these parameters to simulate left-to-right context during fine-tuning.

In [ ]:
# Load the main dataset
df = pd.read_csv("final_dataset.csv")

# Load and prepare the unseen validation set
import re

def preprocess_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

unseen_df = pd.read_csv("unseen_validation_set.csv")
unseen_df["Conversation"] = unseen_df["Conversation"].astype(str).apply(preprocess_text)

# Convert main dataset to HuggingFace Dataset using only the Conversation column
dataset = Dataset.from_pandas(df[["Conversation"]].rename(columns={"Conversation": "text"}))
dataset = {"train": dataset}

# Convert unseen validation set to HuggingFace Dataset
unseen_dataset = Dataset.from_pandas(unseen_df[["Conversation"]].rename(columns={"Conversation": "text"}))


In [3]:
# Step 3: Load XLNet tokenizer and model

tokenizer = XLNetTokenizer.from_pretrained("xlnet-base-cased")
model = XLNetLMHeadModel.from_pretrained("xlnet-base-cased")

/home/shady/Desktop/project/venv/lib64/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [4]:
print(dataset["train"])

Dataset({
    features: ['text'],
    num_rows: 300000
})


In [ ]:
def tokenize_function(example):
    return tokenizer(example["text"])

# Tokenize both the training dataset and unseen validation dataset
tokenized_dataset = dataset["train"].map(tokenize_function, batched=True)
tokenized_unseen = unseen_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 300000/300000 [00:29<00:00, 10269.11 examples/s]


In [6]:
# Grouping function
block_size = 128

def group_texts(examples):
    concatenated = []
    for input_ids in examples["input_ids"]:
        concatenated.extend(input_ids)

    total_length = (len(concatenated) // block_size) * block_size
    input_ids = [concatenated[i : i + block_size] for i in range(0, total_length, block_size)]
    attention_mask = [[1] * block_size for _ in input_ids]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": input_ids.copy(),
    }

In [7]:
# Custom data collator for XLNet to handle perm_mask and target_mapping parameters
from transformers import DataCollatorForLanguageModeling
from dataclasses import dataclass
from typing import Dict, List, Optional, Union

@dataclass
class XLNetDataCollatorForPermutationLanguageModeling:
    """
    Data collator for XLNet permutation language modeling.
    This data collator simulates a left-to-right context during fine-tuning
    by manipulating the perm_mask and target_mapping parameters.
    """
    tokenizer: XLNetTokenizer
    plm_probability: float = 1.0
    max_span_length: int = 5
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, examples: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        batch = self._collate_batch(examples)
        batch_size, sequence_length = batch["input_ids"].size()

        # Create permutation mask
        perm_mask = torch.zeros(
            (batch_size, sequence_length, sequence_length),
            dtype=torch.float,
            device=batch["input_ids"].device
        )

        # Set up causal (left-to-right) attention pattern
        # 1 = cannot attend, 0 = can attend
        for i in range(batch_size):
            for j in range(sequence_length):
                # For each position j, prevent attention to future tokens
                perm_mask[i, j, j+1:sequence_length] = 1.0

        # Create target mapping
        # For each position, we want to predict the token at that position
        target_mapping = torch.zeros(
            (batch_size, sequence_length, sequence_length),
            dtype=torch.float,
            device=batch["input_ids"].device
        )
        
        for i in range(batch_size):
            for j in range(sequence_length):
                # Each position j predicts itself
                target_mapping[i, j, j] = 1.0

        # Add to batch
        batch["perm_mask"] = perm_mask
        batch["target_mapping"] = target_mapping
        
        return batch

    def _collate_batch(self, examples):
        # First, pad all inputs to the same length
        input_ids = [example["input_ids"] for example in examples]
        attention_mask = [example["attention_mask"] for example in examples]
        labels = [example["labels"] for example in examples]
        
        # Convert to tensors
        input_ids = torch.tensor(input_ids, dtype=torch.long)
        attention_mask = torch.tensor(attention_mask, dtype=torch.long)
        labels = torch.tensor(labels, dtype=torch.long)
        
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [ ]:
# Step 4: Training in 10k chunks with evaluation on unseen validation set
chunk_size = 10000
num_chunks = math.ceil(len(df) / chunk_size)
unseen_accuracies = []  # Track accuracy on unseen validation set

# Prepare the unseen validation dataset for evaluation
lm_unseen_dataset = tokenized_unseen.map(group_texts, batched=True, remove_columns=tokenized_unseen.column_names)

# Create custom data collator for XLNet
data_collator = XLNetDataCollatorForPermutationLanguageModeling(tokenizer=tokenizer)

for chunk_idx in range(num_chunks):
    print(f"Training chunk {chunk_idx + 1}/{num_chunks}...")

    chunk_df = df.iloc[chunk_idx * chunk_size : (chunk_idx + 1) * chunk_size]
    dataset = Dataset.from_pandas(chunk_df[["Conversation"]].rename(columns={"Conversation": "text"}))
    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    lm_dataset = tokenized_dataset.map(group_texts, batched=True, remove_columns=tokenized_dataset.column_names)

    # Use the entire chunk for training instead of splitting
    train_dataset = lm_dataset

    training_args = TrainingArguments(
        output_dir=f"temp/xlnet-hinglish-chunk{chunk_idx + 1}",
        evaluation_strategy="epoch",
        num_train_epochs=3,
        per_device_train_batch_size=8,
        save_steps=500,
        save_total_limit=2,
        logging_steps=100,
        warmup_steps=100,
        weight_decay=0.01,
        fp16=True,
        overwrite_output_dir=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=lm_unseen_dataset,  # Use unseen validation set for evaluation
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    trainer.train()

    # Evaluate on unseen validation set and store accuracy
    eval_result = trainer.evaluate()
    print(f"Unseen validation set - Loss: {eval_result['eval_loss']:.4f}")
    if "eval_loss" in eval_result:
        unseen_accuracy = 1 - eval_result["eval_loss"]  # Approximate accuracy
        unseen_accuracies.append(unseen_accuracy)
        print(f"Unseen validation set - Approximate accuracy: {unseen_accuracy:.4f}")

    # Save intermediate model
    model.save_pretrained(f"temp/xlnet-hinglish-chunk{chunk_idx + 1}")
    tokenizer.save_pretrained(f"temp/xlnet-hinglish-chunk{chunk_idx + 1}")

# Step 5: Final model save
model.save_pretrained("./xlnet-hinglish-final")
tokenizer.save_pretrained("./xlnet-hinglish-final")

# Print final accuracy on unseen validation set
print(f"Final XLNet accuracy on unseen validation set: {unseen_accuracies[-1]:.4f}")

Training chunk 1/30...


Map: 100%|██████████| 10000/10000 [00:00<00:00, 58952.65 examples/s]
/home/shady/Desktop/project/venv/lib64/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(

/home/shady/Desktop/project/venv/lib64/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_11916/4169904774.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/tmp/ipykernel_11916/4169904774.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,0.008900,0.000218
2,0.002400,0.000063
3,0.000900,0.000058


Training chunk 2/30...


Map: 100%|██████████| 10000/10000 [00:00<00:00, 46769.30 examples/s]
/home/shady/Desktop/project/venv/lib64/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_11916/4169904774.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(

/home/shady/Desktop/project/venv/lib64/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_11916/4169904774.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
generator = pipeline("text-generation", model="./xlnet-hinglish-final", tokenizer=tokenizer)

user_input = input("Enter a prompt: ")
print(f"testing output for the input: {user_input}")
print(generator(user_input))

Device set to use cuda:0


testing output for the input: mai
[{'generated_text': 'mai'}]


: 

## Evaluate on Unseen Validation Set

In [ ]:
# Load the unseen validation set
print("Loading and preprocessing unseen validation set...")
import re

def preprocess_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

unseen_df = pd.read_csv("unseen_validation_set.csv")
unseen_df["Conversation"] = unseen_df["Conversation"].astype(str).apply(preprocess_text)

# Convert to HuggingFace Dataset
unseen_dataset = Dataset.from_pandas(unseen_df[["Conversation"]].rename(columns={"Conversation": "text"}))
tokenized_unseen = unseen_dataset.map(tokenize_function, batched=True)
lm_unseen_dataset = tokenized_unseen.map(group_texts, batched=True, remove_columns=tokenized_unseen.column_names)

# Create a trainer for evaluation
eval_args = TrainingArguments(
    output_dir="./xlnet_eval",
    per_device_eval_batch_size=8,
    report_to="none"
)

eval_trainer = Trainer(
    model=model,
    args=eval_args,
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Evaluate on the unseen validation set
print("Evaluating XLNet on unseen validation set...")
eval_results = eval_trainer.evaluate(eval_dataset=lm_unseen_dataset)
print(f"Unseen Validation Loss: {eval_results['eval_loss']:.4f}")

## Calculate Perplexity on Unseen Validation Set

In [ ]:
import math
import torch
from torch.nn import functional as F

def calculate_xlnet_perplexity(model, tokenizer, texts, device="cuda", max_length=128):
    model.eval()
    total_loss = 0
    total_tokens = 0
    
    # Limit to a subset if the dataset is too large
    max_samples = min(len(texts), 1000)  # Limit to 1000 samples
    texts = texts[:max_samples]
    
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    with torch.no_grad():
        for text in notebook_tqdm.tqdm(texts):
            # Tokenize and convert to tensor
            encodings = tokenizer(text, truncation=True, max_length=max_length, return_tensors="pt")
            input_ids = encodings.input_ids.to(device)
            attention_mask = encodings.attention_mask.to(device)
            
            # Create perm_mask for autoregressive (left-to-right) prediction
            seq_length = input_ids.size(1)
            perm_mask = torch.zeros((1, seq_length, seq_length), device=device)
            for i in range(seq_length):
                perm_mask[0, i, i+1:] = 1.0  # Cannot attend to tokens to the right
            
            # Create target_mapping for each position
            losses = []
            token_count = attention_mask.sum().item()
            
            # For each position, predict the token
            for i in range(1, seq_length):  # Start from 1 to avoid predicting after BOS token
                if attention_mask[0, i].item() == 0:  # Skip padding tokens
                    continue
                    
                target_mapping = torch.zeros((1, 1, seq_length), device=device)
                target_mapping[0, 0, i] = 1.0  # Predict token at position i
                
                # Forward pass
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    perm_mask=perm_mask,
                    target_mapping=target_mapping,
                )
                
                # Get the predicted distribution for the target position
                logits = outputs.logits.squeeze(0)
                target_id = input_ids[0, i].unsqueeze(0)
                
                # Calculate cross-entropy loss manually
                log_probs = F.log_softmax(logits, dim=-1)
                token_loss = F.nll_loss(log_probs, target_id, reduction='sum').item()
                losses.append(token_loss)
            
            if losses:  # Make sure we have some valid predictions
                batch_loss = sum(losses)
                total_loss += batch_loss
                total_tokens += len(losses)
    
    # Calculate average loss and perplexity
    avg_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
    perplexity = math.exp(avg_loss)
    
    return perplexity

# Get text data for perplexity calculation
training_sample = df["Conversation"].astype(str).apply(preprocess_text).tolist()[:500]  # Use a small subset of training data
unseen_texts = unseen_df["Conversation"].tolist()

# Calculate perplexity on training sample
print("Calculating perplexity on training sample...")
train_perplexity = calculate_xlnet_perplexity(model, tokenizer, training_sample)
print(f"XLNet Training Sample Perplexity: {train_perplexity:.4f}")

# Calculate perplexity on unseen validation set
print("Calculating perplexity on unseen validation set...")
unseen_perplexity = calculate_xlnet_perplexity(model, tokenizer, unseen_texts)
print(f"XLNet Unseen Validation Perplexity: {unseen_perplexity:.4f}")

In [ ]:
# Save the perplexity results for later comparison
import json
import os

# Check if there's an existing transformer perplexity results file
transformer_perplexities = {}
if os.path.exists("transformer_perplexity_results.json"):
    try:
        with open("transformer_perplexity_results.json", "r") as f:
            transformer_perplexities = json.load(f)
    except json.JSONDecodeError:
        transformer_perplexities = {}

# Update with XLNet results
if "training" not in transformer_perplexities:
    transformer_perplexities["training"] = {}
if "unseen" not in transformer_perplexities:
    transformer_perplexities["unseen"] = {}

transformer_perplexities["training"]["XLNet"] = float(train_perplexity)
transformer_perplexities["unseen"]["XLNet"] = float(unseen_perplexity)

# Save the updated perplexity results
with open("transformer_perplexity_results.json", "w") as f:
    json.dump(transformer_perplexities, f, indent=4)

print("✅ Saved XLNet perplexity results to transformer_perplexity_results.json")

In [ ]:
# Plot perplexity and accuracy results for XLNet
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# Plot accuracy on unseen validation set
plt.subplot(1, 2, 1)
plt.plot(unseen_accuracies, marker='o')
plt.title('XLNet: Accuracy on Unseen Validation Set')
plt.xlabel('Chunk')
plt.ylabel('Accuracy')
plt.grid(True)

# Plot perplexity comparison
plt.subplot(1, 2, 2)
metrics = ['Training Sample', 'Unseen Validation']
perplexities = [train_perplexity, unseen_perplexity]
colors = ['blue', 'orange']

plt.bar(metrics, perplexities, color=colors)
plt.title('XLNet: Perplexity Comparison (Lower is Better)')
plt.ylabel('Perplexity')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Annotate perplexity values
for i, p in enumerate(perplexities):
    plt.text(metrics[i], p + 1, f"{p:.2f}", ha='center')

plt.tight_layout()
plt.show()